In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------
# Project Path Detection
# -----------------------------

project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_path = project_root / "data" / "Audiobooks_data.csv"

# -----------------------------
# Load Dataset
# -----------------------------

df = pd.read_csv(data_path, header=None)

df.shape

(14084, 12)

In [2]:
from sklearn.model_selection import train_test_split

In [3]:
# -----------------------------
# Feature - Target Separation
# -----------------------------

X = df.iloc[:, 1:-1].to_numpy()
y = df.iloc[:, -1].to_numpy()

X.shape, y.shape

((14084, 10), (14084,))

In [4]:
# -----------------------------
# Train / Validation / Test Split
# -----------------------------

# train + temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# validation + test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

X_train.shape, X_val.shape, X_test.shape

((9858, 10), (2113, 10), (2113, 10))

### Feature Scaling

Model performansını artırmak amacıyla özellikler StandardScaler kullanılarak ölçeklenmiştir.

Scaler yalnızca eğitim verisi üzerinde öğrenilmiş (fit edilmiş),
validation ve test veri setlerine aynı dönüşüm uygulanmıştır.

Bu yaklaşım veri sızıntısını (data leakage) önlemek için uygulanmaktadır.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [6]:
from pathlib import Path
import joblib

# project root zaten notebookta vardı
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# scaler save
scaler_path = processed_dir / "standard_scaler.joblib"

joblib.dump(scaler, scaler_path)

print("Scaler saved:", scaler_path)

Scaler saved: C:\Users\LenovoTuncay\Documents\deep-learning-tensorflow2\data\processed\standard_scaler.joblib


In [7]:
# -----------------------------
# Processed Data Save
# -----------------------------

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

save_path = processed_dir / "audiobooks_splits_scaled.npz"

np.savez_compressed(
    save_path,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

save_path

WindowsPath('C:/Users/LenovoTuncay/Documents/deep-learning-tensorflow2/data/processed/audiobooks_splits_scaled.npz')

In [8]:
# Verify dataset split sizes
print(
    "Train:", X_train.shape,
    "Val:", X_val.shape,
    "Test:", X_test.shape
)

Train: (9858, 10) Val: (2113, 10) Test: (2113, 10)


## Veri Seti Bölme Stratejisi

Model değerlendirmesinde veri sızıntısını (data leakage) önlemek amacıyla
veri seti üç bölüme ayrılmıştır:

- Training Set → Model öğrenmesi
- Validation Set → Hyperparameter ve threshold kararları
- Test Set → Nihai performans ölçümü

Split oranı:

70% Train / 15% Validation / 15% Test

In [9]:
print("Dataset Split Ratio:")

total = len(X_train) + len(X_val) + len(X_test)

print(f"Train: {len(X_train)/total:.2%}")
print(f"Validation: {len(X_val)/total:.2%}")
print(f"Test: {len(X_test)/total:.2%}")

Dataset Split Ratio:
Train: 69.99%
Validation: 15.00%
Test: 15.00%


## Split Sonrası Sınıf Dağılımı Kontrolü

Veri seti bölündükten sonra hedef değişken dağılımının
train, validation ve test setleri arasında korunup korunmadığı kontrol edilmiştir.

Bu adım model değerlendirmesinin güvenilir olması açısından kritiktir.

In [10]:
def class_distribution(y, name):

    dist = pd.Series(y).value_counts(
        normalize=True
    ).sort_index()

    print(f"{name} Distribution")
    print("-" * 30)

    for cls in dist.index:
        print(f"Class {cls}: {dist[cls]:.2%}")

class_distribution(y_train, "Train")
class_distribution(y_val, "Validation")
class_distribution(y_test, "Test")

Train Distribution
------------------------------
Class 0: 84.11%
Class 1: 15.89%
Validation Distribution
------------------------------
Class 0: 84.10%
Class 1: 15.90%
Test Distribution
------------------------------
Class 0: 84.15%
Class 1: 15.85%


## Preprocessing Summary

Bu aşamada veri ön işleme süreci tamamlanmıştır.

Gerçekleştirilen işlemler:

- Veri seti projeye güvenli şekilde yüklenmiştir
- Model girdileri ve hedef değişken ayrılmıştır
- Stratified train / validation / test bölünmesi uygulanmıştır
- Veri sızıntısını önlemek amacıyla ölçekleme yalnızca eğitim verisi üzerinde öğrenilmiştir
- Veri kümeleri arasındaki sınıf dağılımı doğrulanmıştır
- İşlenmiş veri TensorFlow eğitim sürecinde kullanılmak üzere kaydedilmiştir

Oluşturulan çıktı:

`data/processed/audiobooks_splits_scaled.npz`

Bir sonraki aşamada model mimarisi oluşturulacak ve eğitim süreci başlatılacaktır.